In [12]:
import pandas as pd
import numpy as np
import gzip
import csv
from pathlib import Path


files = {
    "RNA_seq": "Data\Raw\TCGA.BRCA.sampleMap_HiSeqV2.gz",
    "miRNA": "Data\Raw\TCGA.BRCA.sampleMap_miRNA_HiSeq_gene.gz",
    "DNA_Methylation": "Data\Raw\TCGA.BRCA.sampleMap_HumanMethylation450.gz",
    
}


# LOAD COMPRESSED DATASET

def load_dataset(file_path):

    separator = detect_separator(file_path)

    df = pd.read_csv(
        file_path,
        sep="\t",
        compression="gzip",
        index_col=0,
        na_values=[
            "NA", "N/A", "na", "n/a",
            "NaN", "nan", "NULL", "null", ""
        ],
        keep_default_na=True,
        skipinitialspace=True,
        low_memory=False
    )

    df = df.replace(
        r"(?i)^\s*(NA|N/A|NaN|NULL)?\s*$",
        np.nan,
        regex=True
    )

    if df.shape[1] == 0:
        raise ValueError(
            f"Dataset has zero columns: {file_path}. "
            "Check file separator and header structure."
        )

    return df, separator


# INSPECT DATASET

def inspect_dataset(name, file_path):

    df, separator = load_dataset(file_path)

    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    # SHAPE

    print(f"Shape: {df.shape}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")

    # IDENTIFY SAMPLE OR PATIENT IDS

    row_ids = pd.Index(df.index.astype(str))
    col_ids = pd.Index(df.columns.astype(str))

    # TCGA sample-level barcodes
    sample_pattern = (
        r"^TCGA-[A-Za-z0-9]{2}-[A-Za-z0-9]{4}-\d{2}"
    )

    # TCGA patient-level barcodes
    patient_pattern = (
        r"^TCGA-[A-Za-z0-9]{2}-[A-Za-z0-9]{4}$"
    )

    row_samples = row_ids.str.match(sample_pattern)
    col_samples = col_ids.str.match(sample_pattern)

    row_patients = row_ids.str.match(patient_pattern)
    col_patients = col_ids.str.match(patient_pattern)

    sample_ids = pd.Index([])
    patient_ids = pd.Index([])

    if row_samples.sum() > col_samples.sum():

        orientation = "Rows = samples; Columns = features"
        sample_ids = row_ids[row_samples]

    elif col_samples.sum() > row_samples.sum():

        orientation = "Rows = features; Columns = samples"
        sample_ids = col_ids[col_samples]

    elif row_patients.sum() > col_patients.sum():

        orientation = "Rows = patients; Columns = variables"
        patient_ids = row_ids[row_patients]

    elif col_patients.sum() > row_patients.sum():

        orientation = "Rows = variables; Columns = patients"
        patient_ids = col_ids[col_patients]

    else:

        orientation = "Unknown — inspect identifiers"

    print(f"\nOrientation: {orientation}")

# TUMOR AND NORMAL SAMPLE COUNTS

    primary_count = None
    normal_count = None
    unique_patients = None

    if len(sample_ids) > 0:

        sample_types = sample_ids.str.slice(13, 15)

        primary_ids = sample_ids[sample_types == "01"]
        normal_ids = sample_ids[sample_types == "11"]

        primary_count = len(primary_ids)
        normal_count = len(normal_ids)

        unique_patients = (
            sample_ids.str.slice(0, 12).nunique()
        )

        unique_primary_patients = (
            primary_ids.str.slice(0, 12).nunique()
        )

        print("\nSAMPLE INFORMATION")
        print("-" * 45)

        print(f"TCGA sample barcodes: {len(sample_ids):,}")
        print(f"Primary tumor samples (01): {primary_count:,}")
        print(f"Solid tissue normal samples (11): {normal_count:,}")
        print(f"Unique patients: {unique_patients:,}")
        print(
            f"Unique primary tumor patients: "
            f"{unique_primary_patients:,}"
        )

    elif len(patient_ids) > 0:

        unique_patients = patient_ids.nunique()

        print("\nPATIENT INFORMATION")
        print("-" * 45)
        print(f"Unique patients: {unique_patients:,}")
        print("Tumor/normal sample counts: N/A (patient-level IDs)")

    else:

        print("\nNo TCGA barcodes detected on either axis.")

# MISSING VALUES

    total_cells = df.size
    missing_values = int(df.isna().sum().sum())

    missing_percentage = (
        missing_values / total_cells * 100
        if total_cells > 0 else 0
    )

    rows_with_missing = int(
        df.isna().any(axis=1).sum()
    )

    columns_with_missing = int(
        df.isna().any(axis=0).sum()
    )

    print("\nMISSING VALUES")
    print("-" * 45)

    print(f"Total cells: {total_cells:,}")
    print(f"Missing values: {missing_values:,}")
    print(f"Missing percentage: {missing_percentage:.2f}%")
    print(f"Rows with missing values: {rows_with_missing:,}")
    print(f"Columns with missing values: {columns_with_missing:,}")

# DATA PREVIEW

    print("\nFIRST 3 ROWS × FIRST 5 COLUMNS")
    print("-" * 45)
    print(df.iloc[:3, :5].to_string())

# SUMMARY

    summary = {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Orientation": orientation,
        "Primary Tumor Samples": primary_count,
        "Normal Tissue Samples": normal_count,
        "Unique Patients": unique_patients,
        "Missing Values": missing_values,
        "Missing %": round(missing_percentage, 2),
        "Rows With Missing": rows_with_missing,
        "Columns With Missing": columns_with_missing
    }

    return summary


# RUN ANALYSIS FOR ALL FOUR DATASETS

summaries = []

for name, path in files.items():

    try:
        summary = inspect_dataset(name, path)
        summaries.append(summary)

    except Exception as e:
        print(f"\nERROR processing {name}: {e}")


# FINAL SUMMARY TABLE

summary_df = pd.DataFrame(summaries)

print("\n" + "=" * 100)
print("FINAL TCGA-BRCA DATASET SUMMARY")
print("=" * 100)

if not summary_df.empty:
    print(summary_df.to_string(index=False))

    # Save report
    output_file = "tcga_dataset_summary.csv"
    summary_df.to_csv(output_file, index=False)

    print(f"\nSummary saved to: {output_file}")

else:
    print("No datasets were processed successfully.")




RNA_SEQ
Shape: (20530, 1218)
Rows: 20,530
Columns: 1,218

Orientation: Rows = features; Columns = samples

SAMPLE INFORMATION
---------------------------------------------
TCGA sample barcodes: 1,218
Primary tumor samples (01): 1,097
Solid tissue normal samples (11): 114
Unique patients: 1,097
Unique primary tumor patients: 1,097

MISSING VALUES
---------------------------------------------
Total cells: 25,005,540
Missing values: 0
Missing percentage: 0.00%
Rows with missing values: 0
Columns with missing values: 0

FIRST 3 ROWS × FIRST 5 COLUMNS
---------------------------------------------
           TCGA-AR-A5QQ-01  TCGA-D8-A1JA-01  TCGA-BH-A0BQ-01  TCGA-BH-A0BT-01  TCGA-A8-A06X-01
sample                                                                                        
ARHGEF10L           9.5074           7.4346           9.3216           9.0198           9.6417
HIF3A               1.5787           3.6607           2.7224           1.3414           0.5819
RNF17               

In [7]:
clin = pd.read_csv("Data/Raw/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix", sep="\t", index_col=0)
print(clin.shape)
[c for c in clin.columns if "PAM50" in c.upper()]

(1247, 193)


['Integrated_Clusters_with_PAM50__nature2012',
 'PAM50Call_RNAseq',
 'PAM50_mRNA_nature2012']

In [10]:
print(clin["PAM50Call_RNAseq"].value_counts(dropna=False))
print(pd.crosstab(clin["PAM50Call_RNAseq"], clin["PAM50_mRNA_nature2012"]))

LumA      434
NaN       291
LumB      194
Basal     142
Normal    119
Her2       67
Name: PAM50Call_RNAseq, dtype: int64
PAM50_mRNA_nature2012  Basal-like  HER2-enriched  Luminal A  Luminal B  \
PAM50Call_RNAseq                                                         
Basal                          91              0          1          0   
Her2                            0             45          0          0   
LumA                            2              3        222         31   
LumB                            0              7          3         92   
Normal                          1              1          2          0   

PAM50_mRNA_nature2012  Normal-like  
PAM50Call_RNAseq                    
Basal                            0  
Her2                             0  
LumA                             1  
LumB                             0  
Normal                           6  


In [13]:
mirna = pd.read_csv("Data\Raw\TCGA.BRCA.sampleMap_miRNA_HiSeq_gene.gz", sep="\t", index_col=0)
print("min value:", mirna.min().min())
print("number of exact zeros:", (mirna == 0).sum().sum())

miss_per_mirna = mirna.isna().mean(axis=1)
print(miss_per_mirna.describe())

print("missing in 0% of samples:", (miss_per_mirna == 0).sum())
print("missing in <20% of samples:", (miss_per_mirna < 0.2).sum())
print("missing in >90% of samples:", (miss_per_mirna > 0.9).sum())

min value: 0.0639
number of exact zeros: 0
count    2238.000000
mean        0.649310
std         0.382379
min         0.000000
25%         0.289062
50%         0.847957
75%         0.972356
max         0.998798
dtype: float64
missing in 0% of samples: 225
missing in <20% of samples: 513
missing in >90% of samples: 969


In [14]:
meth = pd.read_csv("Data\Raw\TCGA.BRCA.sampleMap_HumanMethylation450.gz",
                   sep="\t", index_col=0).astype("float32")   

miss_per_probe  = meth.isna().mean(axis=1)   # per CpG probe (row)
miss_per_sample = meth.isna().mean(axis=0)   # per sample (column)

print("per probe:\n", miss_per_probe.describe())
print("probes missing everywhere:", (miss_per_probe == 1).sum())
print("probes complete:", (miss_per_probe == 0).sum())
print("probes missing in <5%:", (miss_per_probe < 0.05).sum())
print("per sample:\n", miss_per_sample.describe())

per probe:
 count    485577.000000
mean          0.185712
std           0.387799
min           0.000000
25%           0.000000
50%           0.000000
75%           0.001126
max           1.000000
dtype: float64
probes missing everywhere: 89512
probes complete: 363870
probes missing in <5%: 393966
per sample:
 count    888.000000
mean       0.185712
std        0.001714
min        0.184414
25%        0.184982
50%        0.185149
75%        0.185683
max        0.198652
dtype: float64


In [15]:
def tumour(cols):
    return [s for s in cols if s[13:15] == "01"]

for name, df in {"expr": expr, "mirna": mirna, "meth": meth}.items():
    t = tumour(df.columns)
    n_patients = len({s[:12] for s in t})   
    print(f"{name}: {len(t)} tumour samples, {n_patients} unique patients")


common = set(tumour(expr.columns)) & set(tumour(mirna.columns)) & set(tumour(meth.columns))


labels = clin.loc[clin.index.intersection(list(common)), "PAM50Call_RNAseq"].dropna()
print("all 3 omics:", len(common))
print("all 3 omics + PAM50:", len(labels))
print(labels.value_counts())

expr: 1097 tumour samples, 1097 unique patients
mirna: 749 tumour samples, 749 unique patients
meth: 785 tumour samples, 785 unique patients
all 3 omics: 617
all 3 omics + PAM50: 379
LumA      188
LumB       97
Basal      55
Her2       25
Normal     14
Name: PAM50Call_RNAseq, dtype: int64


In [16]:
clin_common = clin.reindex(list(common))


unlabelled = clin_common[clin_common["PAM50Call_RNAseq"].isna()]
print("unlabelled:", len(unlabelled))   

print("recoverable:", unlabelled["PAM50_mRNA_nature2012"].notna().sum())
print(unlabelled["PAM50_mRNA_nature2012"].value_counts())

unlabelled: 238
recoverable: 3
Luminal A        1
HER2-enriched    1
Basal-like       1
Name: PAM50_mRNA_nature2012, dtype: int64


In [17]:
[c for c in clin.columns if "receptor" in c.lower() or "ER_" in c or "HER2" in c.upper()]

['ER_Status_nature2012',
 'HER2_Final_Status_nature2012',
 'breast_carcinoma_estrogen_receptor_status',
 'breast_carcinoma_progesterone_receptor_status',
 'her2_and_centromere_17_positive_finding_other_measuremnt_scl_txt',
 'her2_erbb_method_calculation_method_text',
 'her2_erbb_pos_finding_cell_percent_category',
 'her2_erbb_pos_finding_fluorescence_n_st_hybrdztn_clcltn_mthd_txt',
 'her2_immunohistochemistry_level_result',
 'her2_neu_and_centromere_17_copy_number_analysis_npt_ttl_nmbr_cnt',
 'her2_neu_breast_carcinoma_copy_analysis_input_total_number',
 'her2_neu_chromosone_17_signal_ratio_value',
 'her2_neu_metastatic_breast_carcinoma_copy_analysis_inpt_ttl_nmbr',
 'lab_proc_her2_neu_immunohistochemistry_receptor_status',
 'lab_procedure_her2_neu_in_situ_hybrid_outcome_type',
 'metastatic_breast_carcinoma_estrogen_receptor_detection_mthd_txt',
 'metastatic_breast_carcinoma_estrogen_receptor_status',
 'metastatic_breast_carcinoma_her2_erbb_method_calculatin_mthd_txt',
 'metastatic_bre